# 06 – Robustness Analyses

This notebook tests how stable the results in `02_model_training_and_group_audit.ipynb` are. It reuses the same features, model settings and 0.50 threshold, and adds five checks:

1. **Repeated sampling and splitting** – 30 random seeds instead of one, with 95% intervals.
2. **Survey weights** – predicted and actual rates recomputed with the ACS person weight (`PWGTP`).
3. **Error-rate parity** – true positive and false positive rates by group (equalized odds).
4. **Direct proxy test** – how well the features in Model B predict sex.
5. **Other states and years** – California 2023 and Texas, New York and Florida 2024.

Running every section takes about 30 minutes on a laptop.

## 1. Install packages

Run the cell below if you are using Google Colab or a fresh local environment.

In [1]:
!pip install -q folktables scikit-learn pandas numpy

'pip' is not recognized as an internal or external command,
operable program or batch file.


## 2. Imports and settings

In [2]:
import os
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from folktables import ACSDataSource

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42
N_SEEDS_MAIN = 30       # repeated splits for California 2024
N_SEEDS_REPLICATION = 10  # repeated splits for each other state/year
MAX_ROWS = 60000

# ACS files are large; set ACS_DATA_DIR to keep them outside the repo
DATA_DIR = os.environ.get("ACS_DATA_DIR", "data")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## 3. Load and prepare ACS data

Same preprocessing as notebook 01, with one addition: the person weight `PWGTP` is kept for the weighted audit.

In [3]:
feature_map = {
    "AGEP": "age", "SCHL": "education", "MAR": "marital_status",
    "RELSHIPP": "relationship", "DIS": "disability", "ESP": "employment_status_parent",
    "CIT": "citizenship", "MIG": "mobility_status", "MIL": "military_service",
    "ANC": "ancestry", "NATIVITY": "nativity", "DEAR": "hearing_difficulty",
    "DEYE": "vision_difficulty", "DREM": "cognitive_difficulty", "SEX": "sex", "RAC1P": "race",
}

sex_map = {1: "Male", 2: "Female"}

race_map = {
    1: "White", 2: "Black", 3: "American Indian", 4: "Alaska Native",
    5: "American Indian/Alaska Native - specified tribe", 6: "Asian",
    7: "Native Hawaiian/Pacific Islander", 8: "Other", 9: "Two or more races",
}

citizenship_map = {
    1: "Born in U.S.", 2: "Born in Puerto Rico / U.S. Island Areas",
    3: "Born abroad to U.S. parents", 4: "U.S. citizen by naturalization", 5: "Not a U.S. citizen",
}

numeric_features = ["age", "education"]
categorical_features_with_sex = [
    "marital_status", "relationship", "disability", "employment_status_parent",
    "citizenship_label", "mobility_status", "military_service", "ancestry", "nativity",
    "hearing_difficulty", "vision_difficulty", "cognitive_difficulty", "sex_label", "race_label",
]
categorical_features_no_sex = [c for c in categorical_features_with_sex if c != "sex_label"]


def load_acs(year, state):
    """Load one state-year of ACS PUMS person data, prepared as in notebook 01."""
    data_source = ACSDataSource(survey_year=year, horizon="1-Year", survey="person", root_dir=DATA_DIR)
    acs_data = data_source.get_data(states=[state], download=True)

    df = acs_data[list(feature_map) + ["ESR", "PWGTP"]].copy().rename(columns=feature_map)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[(df["age"] >= 16) & (df["age"] <= 90)].copy()
    df["employment"] = df["ESR"].isin([1, 2, 4, 5]).astype(int)
    df = df.drop(columns=["ESR"])

    df["sex_label"] = df["sex"].map(sex_map)
    df["race_label"] = df["race"].map(race_map)
    df["citizenship_label"] = df["citizenship"].map(citizenship_map)
    return df


df_ca = load_acs("2024", "CA")
print("California 2024 shape:", df_ca.shape)

California 2024 shape: (328738, 21)


## 4. Helper functions

`sample_model_data` reproduces the stratified 60,000-row sample from notebook 01. It is written as an explicit loop because pandas 3 drops the grouping column inside `groupby().apply()`.

`group_metrics` computes, for each group, the unweighted and weighted predicted employment rate, the true positive rate (TPR) and the false positive rate (FPR). DI ratios compare each group with the highest-rate group, as in notebook 02.

In [4]:
def sample_model_data(df, seed):
    if len(df) <= MAX_ROWS:
        return df.copy()
    parts = [
        x.sample(n=min(len(x), max(1, int(MAX_ROWS * len(x) / len(df)))), random_state=seed)
        for _, x in df.groupby("employment")
    ]
    return pd.concat(parts).sample(frac=1, random_state=seed).copy()


def build_pipeline(numeric_cols, categorical_cols, seed):
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("scaler", StandardScaler())]), numeric_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_cols),
        ],
        remainder="drop"
    )
    classifier = RandomForestClassifier(
        n_estimators=150,
        max_depth=9,
        min_samples_leaf=25,
        class_weight="balanced",
        random_state=seed,
        n_jobs=-1
    )
    return Pipeline(steps=[("preprocessor", preprocessor), ("classifier", classifier)])


def weighted_mean(x, w):
    return float(np.sum(x * w) / np.sum(w))


def group_metrics(results, keys, min_n=100):
    """Per-group selection rates (unweighted and PWGTP-weighted), TPR and FPR."""
    rows = []
    for group, d in results.groupby(keys):
        if len(d) < min_n:
            continue
        positives, negatives = d[d["actual"] == 1], d[d["actual"] == 0]
        rows.append({
            "group": group if isinstance(group, str) else ", ".join(group),
            "n": len(d),
            "rate": d["predicted"].mean(),
            "rate_w": weighted_mean(d["predicted"], d["weight"]),
            "actual_w": weighted_mean(d["actual"], d["weight"]),
            "tpr": positives["predicted"].mean(),
            "fpr": negatives["predicted"].mean(),
        })
    table = pd.DataFrame(rows).set_index("group")
    table["di"] = table["rate"] / table["rate"].max()
    table["di_w"] = table["rate_w"] / table["rate_w"].max()
    return table

## 5. One run = sample, split, train Models A and B, audit, proxy test

Each run uses one seed for the sample, the 80/20 split and the random forests, so seed 42 corresponds to the primary analysis in notebook 02.

In [5]:
def run_once(df, seed, proxy_details=False):
    dm = sample_model_data(df, seed)
    for col in categorical_features_with_sex:
        dm[col] = dm[col].astype(str)

    idx_train, idx_test = train_test_split(
        dm.index, test_size=0.20, random_state=seed, stratify=dm["employment"]
    )
    train, test = dm.loc[idx_train], dm.loc[idx_test]

    out = {"seed": seed}
    for tag, cats in [("A", categorical_features_with_sex), ("B", categorical_features_no_sex)]:
        features = numeric_features + cats
        model = build_pipeline(numeric_features, cats, seed).fit(train[features], train["employment"])
        proba = model.predict_proba(test[features])[:, 1]
        pred = (proba >= 0.50).astype(int)

        results = pd.DataFrame({
            "sex": test["sex_label"].values,
            "race": test["race_label"].values,
            "actual": test["employment"].values,
            "predicted": pred,
            "weight": test["PWGTP"].values,
        })
        by_sex = group_metrics(results, "sex", min_n=1)
        by_sex_race = group_metrics(results, ["sex", "race"])

        out[tag] = {
            "accuracy": accuracy_score(test["employment"], pred),
            "di_sex": by_sex.loc["Female", "rate"] / by_sex.loc["Male", "rate"],
            "di_sex_w": by_sex.loc["Female", "rate_w"] / by_sex.loc["Male", "rate_w"],
            "tpr_gap": by_sex.loc["Male", "tpr"] - by_sex.loc["Female", "tpr"],
            "fpr_gap": by_sex.loc["Female", "fpr"] - by_sex.loc["Male", "fpr"],
            "n_below": int((by_sex_race["di"] < 0.80).sum()),
            "n_below_w": int((by_sex_race["di_w"] < 0.80).sum()),
            "groups": by_sex_race,
        }

    # Direct proxy test: how well do Model B's features predict sex?
    features_b = numeric_features + categorical_features_no_sex
    train_sex = (train["sex_label"] == "Female").astype(int)
    test_sex = (test["sex_label"] == "Female").astype(int)
    proxy_model = build_pipeline(numeric_features, categorical_features_no_sex, seed).fit(train[features_b], train_sex)
    out["proxy_auc"] = roc_auc_score(test_sex, proxy_model.predict_proba(test[features_b])[:, 1])

    if proxy_details:
        names = proxy_model.named_steps["preprocessor"].get_feature_names_out()
        importances = pd.Series(proxy_model.named_steps["classifier"].feature_importances_, index=names)
        out["proxy_importance"] = importances.sort_values(ascending=False)

        single = {}
        for col in features_b:
            num = [col] if col in numeric_features else []
            cat = [] if num else [col]
            m = build_pipeline(num, cat, seed).fit(train[[col]], train_sex)
            single[col] = roc_auc_score(test_sex, m.predict_proba(test[[col]])[:, 1])
        out["proxy_single_feature_auc"] = pd.Series(single).sort_values(ascending=False)

    return out


def interval(values):
    """Mean and 95% percentile interval across runs."""
    v = np.asarray(values, dtype=float)
    return v.mean(), np.percentile(v, 2.5), np.percentile(v, 97.5)


def fmt(values, digits=2):
    m, lo, hi = interval(values)
    return f"{m:.{digits}f} [{lo:.{digits}f}, {hi:.{digits}f}]"

## 6. Repeated splits: California 2024

30 runs, seeds 42 to 71. Seed 42 matches notebook 02 closely but not exactly, because library versions differ from the original Colab run.

In [6]:
runs_ca = []
for seed in range(RANDOM_STATE, RANDOM_STATE + N_SEEDS_MAIN):
    r = run_once(df_ca, seed, proxy_details=(seed == RANDOM_STATE))
    runs_ca.append(r)
    print(f"seed {seed}: DI sex A {r['A']['di_sex']:.3f}, B {r['B']['di_sex']:.3f} | "
          f"groups < 0.80 A {r['A']['n_below']}, B {r['B']['n_below']}")

seed 42: DI sex A 0.944, B 0.984 | groups < 0.80 A 3, B 3


seed 43: DI sex A 0.955, B 0.987 | groups < 0.80 A 2, B 1


seed 44: DI sex A 0.955, B 0.995 | groups < 0.80 A 4, B 4


seed 45: DI sex A 0.944, B 0.979 | groups < 0.80 A 3, B 3


seed 46: DI sex A 0.976, B 1.006 | groups < 0.80 A 3, B 4


seed 47: DI sex A 0.933, B 0.986 | groups < 0.80 A 3, B 3


... (20 outputs left out)

seed 68: DI sex A 0.960, B 1.004 | groups < 0.80 A 3, B 2


seed 69: DI sex A 0.948, B 0.992 | groups < 0.80 A 2, B 2


seed 70: DI sex A 0.946, B 0.989 | groups < 0.80 A 4, B 3


seed 71: DI sex A 0.987, B 1.010 | groups < 0.80 A 2, B 4


### 6.1 Overall results across runs

In [7]:
overall = pd.DataFrame({
    tag: {
        "Accuracy": fmt([r[tag]["accuracy"] for r in runs_ca], 3),
        "Female/male DI": fmt([r[tag]["di_sex"] for r in runs_ca], 3),
        "Female/male DI, weighted": fmt([r[tag]["di_sex_w"] for r in runs_ca], 3),
        "Groups < 0.80": fmt([r[tag]["n_below"] for r in runs_ca], 1),
        "Groups < 0.80, weighted": fmt([r[tag]["n_below_w"] for r in runs_ca], 1),
        "Male - female TPR (pts)": fmt([100 * r[tag]["tpr_gap"] for r in runs_ca], 1),
        "Female - male FPR (pts)": fmt([100 * r[tag]["fpr_gap"] for r in runs_ca], 1),
    }
    for tag in ["A", "B"]
}).rename(columns={"A": "Model A: includes sex", "B": "Model B: excludes sex"})

display(overall)

,Model A: includes sex,Model B: excludes sex
Accuracy,"0.780 [0.774, 0.785]","0.779 [0.772, 0.786]"
Female/male DI,"0.953 [0.928, 0.980]","0.991 [0.971, 1.014]"
"Female/male DI, weighted","0.940 [0.916, 0.965]","0.975 [0.951, 1.008]"
Groups < 0.80,"2.8 [1.7, 4.0]","2.9 [1.0, 4.0]"
"Groups < 0.80, weighted","1.4 [0.0, 3.0]","0.8 [0.0, 2.0]"
Male - female TPR (pts),"2.0 [-0.4, 3.9]","-0.1 [-1.6, 1.6]"
Female - male FPR (pts),"5.8 [3.8, 8.3]","8.7 [7.3, 10.9]"


### 6.2 Intersectional DI ratios across runs (manuscript Table 4)

Mean and 95% interval of each group's DI ratio, the share of runs in which it falls below 0.80, and the weighted DI.

In [8]:
def intersectional_summary(runs):
    groups = runs[0]["A"]["groups"].index
    rows = []
    for g in groups:
        row = {"group": g, "mean n": np.mean([r["A"]["groups"].loc[g, "n"] for r in runs])}
        for tag in ["A", "B"]:
            di = [r[tag]["groups"].loc[g, "di"] for r in runs]
            di_w = [r[tag]["groups"].loc[g, "di_w"] for r in runs]
            row[f"Model {tag} DI"] = fmt(di)
            row[f"Model {tag} runs < 0.80"] = f"{100 * np.mean(np.array(di) < 0.80):.0f}%"
            row[f"Model {tag} DI, weighted"] = fmt(di_w)
            row[f"_sort_{tag}"] = np.mean(di)
        rows.append(row)
    table = pd.DataFrame(rows).set_index("group").sort_values("_sort_A", ascending=False)
    return table.drop(columns=["_sort_A", "_sort_B"])


table4 = intersectional_summary(runs_ca)
display(table4)

,mean n,Model A DI,Model A runs < 0.80,"Model A DI, weighted",Model B DI,Model B runs < 0.80,"Model B DI, weighted"
group,,,,,,,
"Male, Asian",1018.400000,"0.99 [0.95, 1.00]",0%,"0.98 [0.94, 1.00]","0.98 [0.93, 1.00]",0%,"0.98 [0.93, 1.00]"
"Male, Other",967.000000,"0.98 [0.94, 1.00]",0%,"0.99 [0.96, 1.00]","0.97 [0.92, 1.00]",0%,"0.99 [0.94, 1.00]"
"Female, Other",947.466667,"0.95 [0.89, 1.00]",0%,"0.93 [0.88, 0.98]","0.99 [0.97, 1.00]",0%,"0.98 [0.94, 1.00]"
"Male, Two or more races",983.366667,"0.95 [0.90, 0.98]",0%,"0.95 [0.90, 0.99]","0.94 [0.90, 0.97]",0%,"0.94 [0.89, 0.99]"
"Female, Asian",1167.300000,"0.94 [0.89, 0.97]",0%,"0.91 [0.87, 0.95]","0.95 [0.92, 0.99]",0%,"0.94 [0.90, 0.97]"
"Female, Two or more races",999.200000,"0.91 [0.87, 0.96]",0%,"0.90 [0.84, 0.97]","0.94 [0.91, 0.98]",0%,"0.94 [0.89, 0.98]"
"Male, White",2537.700000,"0.82 [0.78, 0.84]",17%,"0.85 [0.81, 0.89]","0.81 [0.77, 0.84]",33%,"0.85 [0.81, 0.89]"
"Female, White",2603.033333,"0.76 [0.72, 0.80]",93%,"0.80 [0.75, 0.84]","0.78 [0.74, 0.80]",93%,"0.82 [0.77, 0.85]"
"Female, Black",261.700000,"0.75 [0.69, 0.83]",83%,"0.81 [0.72, 0.91]","0.76 [0.70, 0.84]",77%,"0.83 [0.75, 0.94]"


## 7. Why weighting matters

The unweighted ACS sample is older and less often employed than the population it represents. Institutional group quarters (`RELSHIPP` = 37, for example prisons) are over-represented, especially for Black men.

In [9]:
weight_check = []
for (sex, race), d in df_ca[df_ca["race_label"].isin(["Black", "White"])].groupby(["sex_label", "race_label"]):
    weight_check.append({
        "group": f"{sex}, {race}",
        "employed, unweighted": d["employment"].mean(),
        "employed, weighted": np.average(d["employment"], weights=d["PWGTP"]),
        "institutional GQ, unweighted": (d["relationship"] == 37).mean(),
        "institutional GQ, weighted": np.average(d["relationship"] == 37, weights=d["PWGTP"]),
        "age 65+, unweighted": (d["age"] >= 65).mean(),
        "age 65+, weighted": np.average(d["age"] >= 65, weights=d["PWGTP"]),
    })

print(f"All adults employed: unweighted {df_ca['employment'].mean():.3f}, "
      f"weighted {np.average(df_ca['employment'], weights=df_ca['PWGTP']):.3f}")
display(pd.DataFrame(weight_check).set_index("group").round(3))

All adults employed: unweighted 0.571, weighted 0.609


,"employed, unweighted","employed, weighted","institutional GQ, unweighted","institutional GQ, weighted","age 65+, unweighted","age 65+, weighted"
group,,,,,,
"Female, Black",0.499,0.560,0.033,0.010,0.289,0.210
"Female, White",0.501,0.540,0.016,0.006,0.346,0.284
"Male, Black",0.469,0.569,0.143,0.051,0.231,0.165
"Male, White",0.582,0.629,0.023,0.010,0.310,0.250


## 8. Error-rate parity by sex × race (seed 42)

TPR = share of employed people predicted employed. FPR = share of people not employed who are predicted employed. Equalized odds requires both to be equal across groups.

In [10]:
r42 = runs_ca[0]
error_rates = pd.concat(
    {f"Model {tag}": r42[tag]["groups"][["n", "tpr", "fpr"]] for tag in ["A", "B"]},
    axis=1
)
display(error_rates.round(3))

Model A               Model B              
                                n    tpr    fpr       n    tpr    fpr
group                                                                
Female, Asian                1230  0.869  0.379    1230  0.880  0.389
Female, Black                 258  0.840  0.237     258  0.840  0.252
Female, Other                 952  0.815  0.421     952  0.848  0.461
Female, Two or more races     967  0.805  0.369     967  0.820  0.396
Female, White                2575  0.796  0.252    2575  0.815  0.271
Male, Asian                  1022  0.887  0.319    1022  0.881  0.308
Male, Black                   256  0.805  0.227     256  0.797  0.227
Male, Other                   955  0.893  0.303     955  0.890  0.303
Male, Two or more races       974  0.853  0.319     974  0.848  0.307
Male, White                  2563  0.814  0.224    2563  0.809  0.219

## 9. Direct proxy test (seed 42)

A random forest with Model B's settings predicts sex from Model B's features. AUC of 0.50 means no information about sex; 1.00 means sex is fully recoverable.

In [11]:
print("Proxy AUC across runs:", fmt([r["proxy_auc"] for r in runs_ca]))

print("\nAUC using one feature at a time:")
display(r42["proxy_single_feature_auc"].round(3).to_frame("AUC"))

print("Top 10 encoded features in the proxy model:")
display(r42["proxy_importance"].head(10).round(4).to_frame("importance"))

Proxy AUC across runs: 0.68 [0.67, 0.69]

AUC using one feature at a time:


,AUC
relationship,0.570
marital_status,0.561
military_service,0.545
education,0.539
age,0.526
citizenship_label,0.520
nativity,0.517
race_label,0.513
hearing_difficulty,0.509
mobility_status,0.506


Top 10 encoded features in the proxy model:


,importance
cat__military_service_4.0,0.2232
cat__military_service_2.0,0.1578
cat__marital_status_2,0.1173
cat__relationship_21,0.0715
num__age,0.0523
cat__marital_status_1,0.0482
cat__relationship_37,0.0403
cat__marital_status_5,0.0349
num__education,0.0338
cat__relationship_20,0.0326


## 10. Other states and years

10 runs each. California 2023 checks a second year; Texas, Florida and New York are the next three most populous states.

In [12]:
replication_sets = [("2023", "CA"), ("2024", "TX"), ("2024", "NY"), ("2024", "FL")]
runs_by_dataset = {"CA 2024": runs_ca}

for year, state in replication_sets:
    df_state = load_acs(year, state)
    runs = [run_once(df_state, seed) for seed in range(RANDOM_STATE, RANDOM_STATE + N_SEEDS_REPLICATION)]
    runs_by_dataset[f"{state} {year}"] = runs
    print(f"{state} {year}: done ({len(df_state):,} records)")

CA 2023: done (326,241 records)


TX 2024: done (237,632 records)


NY 2024: done (176,828 records)


FL 2024: done (182,735 records)


### 10.1 Replication summary (manuscript Table 5)

In [13]:
def count_range(values):
    v = np.asarray(values)
    return f"{v.mean():.1f} ({v.min()}–{v.max()})"


table5 = pd.DataFrame({
    name: {
        "Runs": len(runs),
        "Female/male DI, A": f"{np.mean([r['A']['di_sex'] for r in runs]):.3f}",
        "Female/male DI, B": f"{np.mean([r['B']['di_sex'] for r in runs]):.3f}",
        "Groups < 0.80, A": count_range([r["A"]["n_below"] for r in runs]),
        "Groups < 0.80, B": count_range([r["B"]["n_below"] for r in runs]),
        "Weighted groups < 0.80, A": count_range([r["A"]["n_below_w"] for r in runs]),
        "Weighted groups < 0.80, B": count_range([r["B"]["n_below_w"] for r in runs]),
        "Female - male FPR, A (pts)": f"{100 * np.mean([r['A']['fpr_gap'] for r in runs]):.1f}",
        "Female - male FPR, B (pts)": f"{100 * np.mean([r['B']['fpr_gap'] for r in runs]):.1f}",
        "Proxy AUC": f"{np.mean([r['proxy_auc'] for r in runs]):.2f}",
    }
    for name, runs in runs_by_dataset.items()
}).T

display(table5)

,Runs,"Female/male DI, A","Female/male DI, B","Groups < 0.80, A","Groups < 0.80, B","Weighted groups < 0.80, A","Weighted groups < 0.80, B","Female - male FPR, A (pts)","Female - male FPR, B (pts)",Proxy AUC
CA 2024,30,0.953,0.991,2.8 (1–4),2.9 (1–4),1.4 (0–3),0.8 (0–2),5.8,8.7,0.68
CA 2023,10,0.955,1.000,3.2 (2–4),3.6 (3–4),0.9 (0–2),0.5 (0–2),6.2,9.8,0.68
TX 2024,10,0.957,1.016,5.3 (4–7),4.2 (4–5),3.6 (1–6),1.7 (0–4),9.3,13.6,0.69
NY 2024,10,0.982,0.990,2.9 (2–4),2.8 (1–4),0.2 (0–2),0.3 (0–2),3.9,4.2,0.67
FL 2024,10,1.004,1.016,3.6 (3–4),3.6 (3–4),2.2 (2–3),2.1 (2–3),7.5,8.2,0.71


## Key takeaways

- **The sex-only check always passes.** Across every run, state and year, the female/male DI ratio stays well above 0.80 for both models.
- **Intersectional failures persist in the unweighted data.** One to seven sex × race groups fall below 0.80 in every dataset, with or without sex in the model.
- **Which groups fail depends on sampling and weighting.** With ACS person weights, fewer groups fall below 0.80, and the Black groups' intervals are wide. Intersectional audits need sample sizes and confidence intervals, not a single pass/fail number.
- **Removing sex moves the error gap.** True positive rates by sex become nearly equal, but the false positive gap between women and men widens in all five datasets.
- **The remaining features proxy for sex.** Model B's features predict sex with an AUC of about 0.67–0.71, mostly through household relationship, marital status and military service.